# 01 · Exploratory analysis: how fast is the shift to BEVs?

**Business question:** How fast is Sweden shifting to battery-electric cars among private buyers, where is it fastest or slowest and why, and what BEV share is plausible by 2028?

This notebook covers the first part, **how fast**. It describes the national trend, compares private buyers with company and dealer registrations, measures the speed of change and checks for seasonality. Regional differences (02), the climate-bonus effect (03) and the forecast (04) build on what is found here.

**Metrics (see `CLAUDE.md`)**
- **BEV share** = BEV new registrations ÷ all new passenger-car registrations (all eight fuel types). Every share over more than one month is a *ratio of sums*, never an average of monthly shares.
- **Private buyers** (`owner_type = private`) are the headline population. They're only available from Jan 2021. The all-owner series (2006 onwards) is long-run context and is always labelled *all owners*.
- **PHEV share** is reported alongside BEV but never added to it.
- The climate bonus ended on **8 Nov 2022**.
- Data runs through **Aug 2026**, so 2026 is always a partial year (Jan–Aug).

## 1. Setup and data checks

**Choices made here**
- Read the processed CSVs rather than SQLite. They hold the same tables, and pandas reads CSV directly. The SQLite copy is for the `sql/` phase and Power BI.
- The county fact table leaves out zero cells (see `SOURCES.md`), so the pivot fills missing owner × fuel × month combinations with **0**. If they were dropped instead, any month with zero registrations for a fuel type would vanish.
- Chart style is defined once: thin lines, light gridlines, BEV in blue and PHEV in orange. When private buyers are compared with other owner types, private is the only coloured line and the others are grey, so the headline population stands out.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC, FIGS = ROOT / "data" / "processed", ROOT / "docs" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

FUELS = ["BEV", "PHEV", "HEV", "PETROL", "DIESEL", "ETHANOL", "GAS", "OTHER"]
BONUS_END = pd.Timestamp("2022-11-08")

# Chart style: validated default palette slots 1-2, recessive chrome
BLUE, ORANGE, RED = "#2a78d6", "#eb6834", "#e34948"
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
GREY_DARK, GREY_LIGHT, SURFACE = "#6f6e69", "#b5b3ab", "#fcfcfb"
mpl.rcParams.update({
    "figure.figsize": (10, 4.2), "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 12,
    "axes.labelcolor": INK2, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "ytick.left": False, "lines.linewidth": 2, "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"],
})
PCT = mpl.ticker.PercentFormatter(1.0, decimals=0)

def mark_bonus(ax, y=0.97):
    ax.axvline(BONUS_END, color=AXIS, lw=1, zorder=0)
    ax.annotate("Bonus ends 8 Nov 2022", (BONUS_END, y), xycoords=("data", "axes fraction"),
                xytext=(4, 0), textcoords="offset points", fontsize=8.5, color=INK2, va="top")

def label_end(ax, s, text, dy=0):
    s = s.dropna()
    ax.annotate(text, (s.index[-1], s.iloc[-1]), xytext=(6, dy), textcoords="offset points",
                fontsize=9, color=INK2, va="center")

def save(fig, name):
    fig.savefig(FIGS / f"{name}.png")

In [2]:
nat = pd.read_csv(PROC / "national_monthly_by_fuel.csv", parse_dates=["month"]).set_index("month")
fact = pd.read_csv(PROC / "fact_registrations_county_monthly.csv", parse_dates=["month"],
                   dtype={"county_code": str})

months = pd.date_range(fact.month.min(), fact.month.max(), freq="MS")
own = (fact.pivot_table(index=["owner_type", "month"], columns="fuel_type", values="registrations",
                        aggfunc="sum", fill_value=0)
           .reindex(columns=FUELS, fill_value=0))
own["total"] = own[FUELS].sum(axis=1)
by_owner = {o: own.loc[o].reindex(months, fill_value=0) for o in ["private", "company", "dealer"]}
priv = by_owner["private"]
LAST = months[-1]

print(f"National series: {nat.index.min():%Y-%m} → {nat.index.max():%Y-%m}  ({len(nat)} months, "
      f"gaps: {len(pd.date_range(nat.index.min(), nat.index.max(), freq='MS')) - len(nat)})")
print(f"County fact:     {months[0]:%Y-%m} → {LAST:%Y-%m}  ({fact.month.nunique()} of {len(months)} months present)")

National series: 2006-01 → 2026-08  (248 months, gaps: 0)
County fact:     2021-01 → 2026-08  (68 of 68 months present)


In [3]:
# Check 1: the county fact table sums to the national t10036 series in every overlapping month
val = pd.read_csv(PROC / "_validation_national_vs_county.csv")
print("Max |national - county sum| difference:", val.abs_pct_diff.max())

# Check 2: the private BEV share rebuilt here matches the column produced by build_dataset.py
rebuilt = (priv.BEV / priv.total).round(5)
print("Private BEV share matches build output:", np.allclose(rebuilt, nat.loc[months, "private_bev_share"]))

# Owner mix: what fraction of all new registrations each buyer type accounts for
mix = pd.DataFrame({o: d.total for o, d in by_owner.items()})
mix_year = mix.groupby(mix.index.year).sum()
(mix_year.div(mix_year.sum(axis=1), axis=0)
         .rename(index={LAST.year: f"{LAST.year} (Jan–{LAST:%b})"})
         .style.format("{:.1%}").set_caption("Share of all new registrations by owner type"))

Max |national - county sum| difference: 0.0
Private BEV share matches build output: True


,private,company,dealer
2021,39.6%,41.1%,19.3%
2022,41.9%,42.3%,15.8%
2023,30.9%,48.5%,20.6%
2024,32.4%,45.4%,22.1%
2025,35.2%,43.1%,21.7%
2026 (Jan–Aug),37.6%,44.0%,18.4%


**What the checks show.** The county fact table matches the national series exactly in every month, and the private BEV share rebuilt here matches the build output. Both data paths are consistent.

The owner mix moves a lot, and that matters for everything below. Private buyers were about **40%** of the market in 2021–22, fell to **31%** in 2023, the first year without the bonus, and have recovered to about **38%** in 2026. Companies (mostly leasing) are the largest group. Because the mix shifts, the all-owner BEV share can move even when no buyer type changes its behaviour. That's one more reason to keep private buyers separate.